# 05 Ablation Sweeps

7가지 SDD 컴포넌트를 하나씩 제거하면서 FID와 linear probe accuracy를 비교합니다.

각 variant는 독립 실행이므로 GPU를 충분히 사용하세요.

In [ ]:
import torch
from src.experiments import load_cfg, deep_update

# ── GPU 자동 감지 ───────────────────────────────────────────────
n_gpu = torch.cuda.device_count()
print(f"감지된 GPU 수: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory // 1024**3} GB")
if n_gpu == 0:
    print("  (GPU 없음 — CPU로 실행됩니다)")


In [ ]:
from src.experiments import load_cfg, deep_update, launch_train
import pandas as pd

base_cfg = load_cfg("configs/cifar10.yaml")
base_cfg = deep_update(base_cfg, {
    "wandb": {"enabled": True, "tags": ["cifar10", "ablation"]},
})

variants = {
    "full_sdd":          {"sdd": {"enabled": True,  "use_centering": True,  "use_sharpening": True,  "use_gating": True,  "use_projection_head": True}},
    "mse_only":          {"sdd": {"enabled": False, "use_centering": False, "use_sharpening": False, "use_gating": False, "use_projection_head": False, "lambda_distill": 0.0}},
    "ema_only":          {"sdd": {"enabled": True,  "use_centering": False, "use_sharpening": False, "use_gating": False, "use_projection_head": False}},
    "no_centering":      {"sdd": {"enabled": True,  "use_centering": False, "use_sharpening": True,  "use_gating": True,  "use_projection_head": True}},
    "no_sharpening":     {"sdd": {"enabled": True,  "use_centering": True,  "use_sharpening": False, "use_gating": True,  "use_projection_head": True}},
    "no_gating":         {"sdd": {"enabled": True,  "use_centering": True,  "use_sharpening": True,  "use_gating": False, "use_projection_head": True}},
    "no_projection_head":{"sdd": {"enabled": True,  "use_centering": True,  "use_sharpening": True,  "use_gating": True,  "use_projection_head": False}},
    "soft_gating":       {"sdd": {"enabled": True,  "use_centering": True,  "use_sharpening": True,  "use_gating": True,  "gating": {"mode": "soft"}}},
}

rows = []
for name, overrides in variants.items():
    cfg = deep_update(base_cfg, overrides)
    cfg["run_name"] = f"cifar10_{name}"
    print(f"\n▶ 학습 중: {name}")
    launch_train(cfg, num_processes=None, with_eval=True)

    from pathlib import Path
    import pandas as _pd
    csv = Path(f"outputs/logs/cifar10_{name}_history.csv")
    if csv.exists():
        df = _pd.read_csv(csv)
        last = df.dropna(subset=["val_fid"]).iloc[-1] if "val_fid" in df.columns else df.iloc[-1]
        rows.append({"variant": name,
                     "fid":              last.get("val_fid"),
                     "linear_probe_acc": last.get("val_linear_probe_acc")})

results = pd.DataFrame(rows)
results


In [ ]:
import matplotlib.pyplot as plt

if not results.empty and "fid" in results.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    r = results.dropna(subset=["fid"]).sort_values("fid")
    axes[0].barh(r["variant"], r["fid"], color="steelblue")
    axes[0].set_xlabel("FID ↓"); axes[0].set_title("FID by variant")

    r2 = results.dropna(subset=["linear_probe_acc"]).sort_values("linear_probe_acc", ascending=False)
    axes[1].barh(r2["variant"], r2["linear_probe_acc"], color="darkorange")
    axes[1].set_xlabel("Linear probe acc ↑"); axes[1].set_title("Probe acc by variant")
    plt.tight_layout()
    plt.savefig("outputs/figures/ablation_summary.png", dpi=150)
    plt.show()
    print(results.sort_values("fid").to_string(index=False))
